# 01 — Refugia raster, grid, nodes → **CHECK STOP 1** · variant surface → **CHECK STOP 2**

First of five notebooks (spec: `spec/wolverine_refugia_connectivity_spec.md` — read it before
touching anything here). Everything that happens BEFORE a run dir exists: the raster characterised,
the cost surface and the refugia classes warped onto the Y2Y-wide 300 m routing grid (**G2**,
**GW1**), the graph self-test (**G4**), the nodes vectorised (**W2/W9**, `cc.node_patches`), and
the "generic terrain rules withheld" variant surface derived (**W1**, **GW2**).

**Two hard stops (W10):** after the node map (check stop 1) and after the variant maps (check
stop 2). Notebook 02 solves ONLY the variant surface, and only after both checks pass.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

## 1 · The input raster — what it is

`input_data/wolverine_refugia/baseline_wolverine_climate_refugia.tif`: EPSG:4326, ~250 m, values
{0, 1, 2, 10, 11, 12}. **2 | 12 = core refugia, 1 | 11 = marginal, 0 | 10 = none.** The +10 offset is
a MODEL SEAM: 10/11/12 come from the northern model (north of ~53.97 °N), 0/1/2 from the southern
one. Decision W4: one layer, core = {2, 12}, marginal = {1, 11}; disclosed in the spec.

In [ ]:
import rasterio
from rasterio import features
import geopandas as gpd
src_path = CFG["nodes"]["raster"]["source"]
with rasterio.open(src_path) as s:
    arr = s.read(1); tr = s.transform; crs = s.crs
    print(f"{src_path.name}: {s.width} x {s.height} @ {abs(tr.a):.6f}° ({s.crs}), dtype {s.dtypes[0]}, nodata {s.nodata}")
valid = ~np.isnan(arr)
y2y = gpd.read_file(config.CORRIDOR_REF).to_crs(crs)
inside = features.rasterize([(g, 1) for g in y2y.geometry], out_shape=arr.shape, transform=tr, dtype="uint8").astype(bool)
lat = tr.f + tr.e * (np.arange(arr.shape[0]) + 0.5)
px_km2 = (abs(tr.a) * 111.32 * np.cos(np.deg2rad(lat))) * (abs(tr.e) * 110.57)
rows = []
for v in [0, 1, 2, 10, 11, 12]:
    m = (arr == v)
    km2 = float((m.sum(axis=1) * px_km2).sum()); km2_in = float(((m & inside).sum(axis=1) * px_km2).sum())
    r = np.where(m.any(axis=1))[0]
    rows.append(dict(value=v, role={0: "none (south model)", 1: "marginal (south)", 2: "CORE (south)", 10: "none (north model)",
                                    11: "marginal (north)", 12: "CORE (north)"}[v],
                     km2=round(km2), km2_inside_y2y=round(km2_in), lat_min=round(float(lat[r.max()]), 2) if len(r) else None,
                     lat_max=round(float(lat[r.min()]), 2) if len(r) else None))
tbl = pd.DataFrame(rows)
print(f"valid pixels {valid.sum():,}; Y2Y coverage {100*(valid & inside).sum()/inside.sum():.2f}%")
tbl

## 2 · Warp the cost surface (**G2**) and the refugia classes (**GW1**) onto the Y2Y-wide 300 m grid

The grid is `config.study_area(20 km)` snapped to 300 m, no latitude cut (`grid.region_filter = None`).
Own namespace `input_data/corridors_300m_y2y/` — the northern warp and its CWD cache are untouched.
`-r near` for both rasters (ordinal classes / categorical classes preserved exactly).

In [ ]:
g = cp.grid(KEY)
cp.warp(g)
chk = cp.check(g)
cp.warp_classes(g, KEY)
cls_chk = cp.check_classes(g, KEY)

## 3 · Graph-layer self-test (**G4**)

In [ ]:
cg.selftest()

## 4 · Nodes — vectorise the core refugia patches (**W2 / W9**)

8-connected components of the core classes on the routing grid, kept at or above `node_min_km2`
(500 km²), numbered **north → south**, auto-named from the PA with the largest overlap (else the
nearest PA + bearing). Writes the H7-FORMAT `node_parts.csv` / `node_parts.gpkg` (every figure reads
the gpkg), `node_names.csv` (fill `display_name` if you want to rename a patch) and
`refugia_summary.json` into the git-tracked `audit/audit_objects/`. No review file: nothing to sign.

In [ ]:
A = cc.node_patches(KEY)
summ = json.loads((HERE / "audit" / "audit_objects" / "refugia_summary.json").read_text())
print("ladder (patches >= km2):", summ["ladder"], "| node land", f"{summ['node_land_km2']:,} km²", "| seam split (core km²):",
      {k: (summ["class_km2_north_of_seam"][k], summ["class_km2_south_of_seam"][k]) for k in ("2", "12")})
pd.set_option("display.max_rows", 100, "display.width", 200)
A.node_table[["node_id", "name", "area_km2", "pa_overlap_name", "pa_overlap_frac", "nearest_pa", "nearest_pa_km", "bearing", "share_north_model", "lat", "lon"]]

## ⛔ CHECK STOP 1 — the raster → node transformation (Ethan)

Review the map below and the table above: does every numbered patch look like a unit worth
connecting, is the 500 km² floor right, are the auto-names sensible? For GIS inspection:
`audit/audit_objects/node_parts.gpkg` (the patches, numbered) and
`input_data/corridors_300m_y2y/refugia_class.tif` (the warped classes). To rename a patch, fill
`display_name` in `audit/audit_objects/node_names.csv` (the run copies + hash-pins it). Then
commit the four audit files.

In [ ]:
fig = cc.node_map(A, n_insets=3, win_km=350, save=HERE / "figures" / "check1_nodes.png") if (HERE / "figures").mkdir(parents=True, exist_ok=True) is None else None
print("GIS: ", HERE / "audit" / "audit_objects" / "node_parts.gpkg")
print("     ", g["dir"] / CFG["nodes"]["raster"]["out_name"])

## 5 · The variant surface (**W1**) — data the variant needs

The source's cost-1000 class bundles three GENERIC-FAUNA terrain rules (elevation > 2,300 m,
slope > 30°, glaciers; Pither et al. 2023 S1 Table) with the human and water layers. The variant
withholds the three terrain rules and retakes the per-pixel maximum. The repo holds the published
surface, not the source's 23 layers, so the retained layers are reproduced by proxy: HydroLAKES
(≥ 10 ha), HydroRIVERS (> 28 m³/s), ocean, and the 90 m gHM (human layers, threshold τ calibrated
below). Glaciers come from RGI 7.0. `data/acquire.py` lists what is needed, where it goes, and
downloads what it can (RGI may need a manual download — it prints the landing page).

In [ ]:
sys.path.insert(0, str(HERE / "data"))
import acquire; importlib.reload(acquire)
status = acquire.check()
DOWNLOAD = False          # set True to let the script fetch the missing open datasets (HydroSHEDS ~1.7 GB, RGI ~0.5 GB)
if DOWNLOAD:
    acquire.fetch()
    status = acquire.check()
assert all(status.values()), "variant inputs missing -- see the table above (data/acquire.py)"

## 6 · Terrain + retained layers → τ → the variant (**GW2**)

`cp.terrain_layers` rasterizes each layer onto the routing grid (cached under
`corridors_300m_y2y/variant_layers/`) and prints how much of the cost-1000 class each explains.
`cp.calibrate_human_tau` picks τ on the 90 m gHM maximum that best separates the human cost-1000
cells (those explained by NO natural layer) from the rest — reported with its confusion table,
never asserted. `cp.derive_variant` then writes `movement_cost_terrain_withheld.tif` + its meta.

In [ ]:
VARIANT = CFG["headline_variant"]
vres = CFG["variants"][VARIANT]["resistance"]; vcfg = vres["variant"]; OUT_NAME = vres["out_name"]
L = cp.terrain_layers(g, vcfg)
conf = cp.calibrate_human_tau(L, g["res_m"])
meta = cp.derive_variant(g, vcfg, L, conf["tau"], OUT_NAME, KEY)
meta["human_tau_calibration"] = conf
(g["dir"] / f"{OUT_NAME}.meta.json").write_text(json.dumps(meta, indent=2))
under = cp.variant_summary_under_nodes(g, L, OUT_NAME, KEY)
cp.check(dict(g, dst=g["dir"] / OUT_NAME))         # G2 on the variant: same grid, classes still {1, 10, 100, 1000}

## ⛔ CHECK STOP 2 — the updated resistance layer (Ethan)

Left: the published surface. Middle: the variant. Right: what changed, one colour per withheld
rule, plus the terrain cells KEPT at 1000 because a retained layer (lake / river / ocean / human
proxy) covers them. The numbers above (km² withheld per rule, by elevation band, under the core
refugia, τ + confusion) are in `input_data/corridors_300m_y2y/movement_cost_terrain_withheld.tif.meta.json`.
For GIS: the surface itself, beside `movement_cost.tif` (published) and `variant_layers/*.tif`.

In [ ]:
fig2 = cp.variant_maps(g, vcfg, L, OUT_NAME, KEY)
print("variant surface:", g["dir"] / OUT_NAME)
print("meta:           ", g["dir"] / f"{OUT_NAME}.meta.json")

## 7 · Timing probe — one cost-weighted-distance pass on the variant surface

Every runtime estimate in the spec scales from this number (the north measured ~35 s per pass on
16 M cells; the Y2Y-wide grid is 47 M cells). Notebook 02's CWD stage = one pass per node.

In [ ]:
import time
from skimage.graph import MCP_Geometric
with rasterio.open(g["dir"] / OUT_NAME) as s:
    cost = s.read(1)
res = np.where(np.isfinite(cost), cost, np.inf)
biggest = int(A.node_table.sort_values("area_km2", ascending=False).iloc[0]["node_id"])
seeds = [tuple(x) for x in np.argwhere(A.node_id == biggest)]
t0 = time.time(); mcp = MCP_Geometric(res); cum, _ = mcp.find_costs(seeds); dt = time.time() - t0
n = len(A.node_table)
print(f"one pass from node R{biggest:02d} ({len(seeds):,} seed cells): {dt:,.0f} s -> CWD set of {n} nodes ≈ {n*dt/3600:.1f} h; "
      f"network bands (~{int(1.3*n)} edges, early-stop tracebacks) ≈ {0.6*1.3*n*dt/3600:.1f} h")
del mcp, cum, res, cost

## Done — before notebook 02

1. Both check stops reviewed (nodes; variant surface). Rename patches in `node_names.csv` if wanted.
2. Commit `audit/audit_objects/` (node_parts.csv, node_parts.gpkg, node_names.csv, refugia_summary.json).
3. Run `02_baseline.ipynb`. It resolves the config (**G8**), re-checks the warps, creates the run dir on
   the VARIANT surface only, and runs the CWD stage (the long one).